In [4]:
import os
import pandas as pd
import matplotlib.pyplot as plt


os.makedirs("../graphs", exist_ok=True)

noises = [0] + [round(i*0.01, 2) for i in range(1, 11)]

for test in ['testAB', 'testAG', 'test']:
    for noise in noises:
        base_dir = f"../results/noise{noise}"
        
        runs = [f"rand{i}" for i in range(100)]
        
        dfs_qbc = []
        dfs_random = []
        
        for run in runs:
            path = os.path.join(base_dir, run)
            if not os.path.exists(path):
                continue
            for file in os.listdir(path):
                if file.startswith("roc_aucs_qbc"):
                    df = pd.read_csv(os.path.join(path, file), sep="\t")
                    dfs_qbc.append(df)
                elif file.startswith("roc_aucs_random"):
                    df = pd.read_csv(os.path.join(path, file), sep="\t")
                    dfs_random.append(df)
        
        if not dfs_qbc or not dfs_random:
            continue
        
        df_qbc_all = pd.concat(dfs_qbc, ignore_index=True)
        df_random_all = pd.concat(dfs_random, ignore_index=True)
        
        qbc_mean = df_qbc_all.groupby("ags_number")[f"roc_aucs_{test}"].mean()
        random_mean = df_random_all.groupby("ags_number")[f"roc_aucs_{test}"].mean()
        
        plt.figure(figsize=(8, 6))
        
        plt.plot(qbc_mean.index, qbc_mean.values, marker=".", label="QBC", color="blue")
        plt.plot(random_mean.index, random_mean.values, marker=".", label="Random", color="orange")
        
        plt.xlabel("Number of antigens (ags_number)")
        plt.ylabel(f"ROC AUC ({test})")
        plt.title(f"ROC AUC ({test}) vs. ags_number (noise={noise})")
        plt.legend()
        plt.grid(True)
        
        plt.savefig(f"../graphs/roc_auc_vs_ags_{test}_noise{noise}.png", dpi=150)
        plt.close()
